# Experiment C — Full Three-Way Comparison

**Evolutionary Prompt Optimization for LLM-Based Sentiment Analysis**  
**Member 4 — Experimental Design and Baseline Implementation**

---

## Purpose

This notebook collects results from Experiments A, B, and C (GA) and produces the **final controlled comparison table** that Member 5 will use for statistical interpretation.

| Label | Experiment | Source notebook | Method |
|---|---|---|---|
| **A** | Manual prompting | `07_experiment_A_manual_prompts.ipynb` | 8 hand-crafted prompts by Member 2 |
| **B** | Random generation | `08_experiment_B_random_baseline.ipynb` | 36 LLM-generated prompts, no selection |
| **C** | Genetic Algorithm | `05_genetic_algorithm_experiment.ipynb` / `06_multi_seed_experiment.ipynb` | GA runs by Member 3 (seeds 42, 7, 123) |

### What this notebook does
1. Loads the summary JSONs produced by notebooks 07, 08 (Member 4), and 05/06 (Member 3)
2. Runs McNemar's test to check statistical significance on the validation split
3. Produces a merged comparison table (CSV + JSON)
4. Visualises all three approaches side-by-side
5. Saves everything to `experiments/results/experiment_C_comparison/`

> **Important**: this notebook does **not** produce the final statistical interpretation or the research discussion — those are Member 5's responsibility.

### Protocol reminder
- Same LLM, same revision, same greedy decoding for all three experiments
- Same 1 000-example opt sample and 1 000-example val sample (seed 42)
- Validation results are reported once, only after prompt selection on opt
- Test set is never used

## 1. Environment Setup

In [ ]:
import os
import sys
import json
import subprocess
import random
import time
import importlib.util
from pathlib import Path

REPO_URL = "https://github.com/sarrazer24/evolutionary-prompt-optimization.git"
REPO_DIR = Path("/kaggle/working/evolutionary-prompt-optimization")

if not REPO_DIR.exists():
    subprocess.run(
        ["git", "clone", "--branch", "main", "--single-branch", REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    if not (REPO_DIR / ".git").exists():
        raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository.")
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only", "origin", "main"], check=True)

os.chdir(REPO_DIR)

REQUIRED_PACKAGES = {
    "transformers": "transformers",
    "accelerate": "accelerate",
    "datasets": "datasets",
    "sklearn": "scikit-learn",
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "scipy": "scipy",
}
missing_packages = [
    pkg for mod, pkg in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(mod) is None
]
if missing_packages:
    print("Installing missing packages:", missing_packages)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", *missing_packages],
        check=True,
    )
else:
    print("All required packages are already installed.")

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

print("Project directory:", Path.cwd())
print("Python version:", sys.version)
print("Setup complete.")

## 2. Imports and Reproducibility

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from scipy.stats import chi2_contingency
import torch
from datetime import datetime
from IPython.display import display

from src.data.sst2_data import load_saved_splits, stratified_sample
from src.llm.config import ModelConfig
from src.llm.model_interface import LLMInterface
from src.evaluation.evaluation import PredictionCache, evaluate_prompt, save_results

# ── Reproducibility ────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"Global seed: {SEED}")

## 3. GPU Check

In [ ]:
if not torch.cuda.is_available():
    print("⚠  No GPU detected — inference will be very slow on CPU.")
    print("   On Kaggle: Settings → Accelerator → GPU T4 x2.")
else:
    for i in range(torch.cuda.device_count()):
        name = torch.cuda.get_device_name(i)
        mem  = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f"GPU {i}: {name}  ({mem:.1f} GB)")

## 4. Configuration

In [ ]:
N_OPT_EXAMPLES = 1_000
N_VAL_EXAMPLES = 1_000

OUTPUT_DIR = Path("experiments/results/experiment_C_comparison")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Paths to the summaries produced by the other notebooks
SUMMARY_A_PATH = Path("experiments/results/experiment_A_manual/experiment_A_summary.json")
SUMMARY_B_PATH = Path("experiments/results/experiment_B_random/experiment_B_summary.json")

# Member 3's GA result directory (seed 42)
GA_SEED42_DIR  = Path("experiments/results/ga_seed_42")   # adjust if needed
GA_MULTI_PATH  = Path("experiments/results/multi_seed_analysis.json")  # from nb06

print("Output directory:", OUTPUT_DIR)

## 5. Load Data

We use the **same sample** as Experiments A and B: seed 42, 1 000 opt examples, 1 000 val examples.

In [ ]:
opt_df, val_df = load_saved_splits()

rng = random.Random(SEED)
opt_sample = stratified_sample(opt_df, n=N_OPT_EXAMPLES, rng=rng)
val_sample  = stratified_sample(val_df,  n=N_VAL_EXAMPLES,  rng=rng)

opt_texts  = opt_sample["text"].tolist()
opt_labels = opt_sample["label"].tolist()
val_texts  = val_sample["text"].tolist()
val_labels = val_sample["label"].tolist()

print(f"Opt sample : {len(opt_texts)} examples")
print(f"Val sample : {len(val_texts)} examples")

## 6. Load the LLM

In [ ]:
model_config = ModelConfig()
llm = LLMInterface(model_config)
llm.load()
print(f"Model loaded: {llm.model_id}")
print(f"Device      : {llm.device}")

## 7. Load Experiment A and B Summaries

In [ ]:
def load_summary(path: Path, label: str) -> dict:
    if not path.exists():
        raise FileNotFoundError(
            f"Summary for Experiment {label} not found at: {path}\n"
            f"Please run notebook {label} first."
        )
    with open(path, encoding="utf-8") as f:
        summary = json.load(f)
    print(f"Loaded Experiment {label}: {path}")
    return summary

summary_A = load_summary(SUMMARY_A_PATH, "A")
summary_B = load_summary(SUMMARY_B_PATH, "B")

print()
print(f"Experiment A — best prompt  : {summary_A['best_prompt_name']}")
print(f"  Opt Macro F1              : {summary_A['best_macro_f1_opt']:.4f}")
print(f"  Val Macro F1              : {summary_A['best_macro_f1_val']:.4f}")
print()
print(f"Experiment B — best candidate: {summary_B['best_candidate_id']}")
print(f"  Opt Macro F1              : {summary_B['best_macro_f1_opt']:.4f}")
print(f"  Val Macro F1              : {summary_B['best_macro_f1_val']:.4f}")

## 8. Load GA Results (Member 3)

We load the best-evolved prompts from Member 3's output files.
- **Primary**: seed 42 from `05_genetic_algorithm_experiment.ipynb`
- **Multi-seed**: seeds 7 and 123 from `06_multi_seed_experiment.ipynb`

If the directories are not found, the cells below will raise a clear error so you can adjust the paths.

In [ ]:
def find_ga_dir(candidates: list[Path]) -> Path:
    """Return the first existing GA output directory."""
    for p in candidates:
        if p.exists() and (p / "best_individual.json").exists():
            return p
    raise FileNotFoundError(
        f"GA output directory not found. Tried:\n"
        + "\n".join(f"  {p}" for p in candidates)
        + "\nPlease run Notebook 05 first or adjust GA_SEED42_DIR."
    )

# Common candidate locations used by Member 3's notebooks
GA_SEED42_CANDIDATES = [
    Path("experiments/results/ga_seed_42"),
    Path("experiments/results/ga_experiment_01"),
    Path("/kaggle/working/genetic_algorithm_experiment_01"),
]

ga_seed42_dir = find_ga_dir(GA_SEED42_CANDIDATES)
print(f"GA seed-42 directory found: {ga_seed42_dir}")

with open(ga_seed42_dir / "best_individual.json", encoding="utf-8") as f:
    ga_best_individual = json.load(f)

ga_best_prompt = ga_best_individual["prompt"]
ga_seed42_opt_f1 = ga_best_individual["fitness"]  # Macro F1 on opt split
print(f"Best evolved prompt (seed 42): {ga_best_prompt[:120]}…")
print(f"Opt Macro F1 (from GA log)   : {ga_seed42_opt_f1:.6f}")

In [ ]:
# ── Multi-seed summary ─────────────────────────────────────────────────────────
GA_MULTI_CANDIDATES = [
    Path("experiments/results/multi_seed_analysis.json"),
    Path("/kaggle/working/multi_seed_analysis.json"),
]

ga_multi = None
for p in GA_MULTI_CANDIDATES:
    if p.exists():
        with open(p, encoding="utf-8") as f:
            ga_multi = json.load(f)
        print(f"Multi-seed analysis found: {p}")
        break

if ga_multi is None:
    print("⚠  Multi-seed analysis not found — will report seed-42 only for GA.")
    ga_seeds = {"seed_42": {"best_macro_f1_opt": ga_seed42_opt_f1}}
else:
    ga_seeds = ga_multi.get("runs", {"seed_42": {"best_macro_f1_opt": ga_seed42_opt_f1}})
    print("GA seeds available:", list(ga_seeds.keys()))

## 9. Validate the Best GA Prompt (Seed 42)

We evaluate the best evolved prompt on the **same validation sample** used by Experiments A and B.  
This allows a direct, like-for-like comparison.

In [ ]:
# Shared prediction cache so identical (prompt, text) pairs are not re-scored.
cache_C = PredictionCache()

print("Evaluating best GA prompt (seed 42) on opt split …")
t0 = time.time()
ga_opt_result = evaluate_prompt(
    prompt=ga_best_prompt,
    examples=opt_texts,
    labels=opt_labels,
    predict_fn=llm.predict_raw_batch,
    model_id=llm.model_id,
    inference_config=llm.inference_config,
    cache=cache_C,
    split_name="opt",
)
opt_elapsed = time.time() - t0
om = ga_opt_result["metrics"]
ga_opt_f1   = om["f1_macro"]
ga_opt_acc  = om["accuracy"]
print(f"  Opt Macro F1 : {ga_opt_f1:.4f}  (from GA log: {ga_seed42_opt_f1:.4f})")
print(f"  Elapsed      : {opt_elapsed:.1f}s")

print("\nEvaluating best GA prompt (seed 42) on val split …")
t0 = time.time()
ga_val_result = evaluate_prompt(
    prompt=ga_best_prompt,
    examples=val_texts,
    labels=val_labels,
    predict_fn=llm.predict_raw_batch,
    model_id=llm.model_id,
    inference_config=llm.inference_config,
    split_name="val",
)
val_elapsed = time.time() - t0
vm_ga = ga_val_result["metrics"]
ga_val_f1  = vm_ga["f1_macro"]
ga_val_acc = vm_ga["accuracy"]
print(f"  Val Macro F1 : {ga_val_f1:.4f}")
print(f"  Elapsed      : {val_elapsed:.1f}s")

## 10. Re-evaluate All Three Best Prompts on the Validation Split

To allow **per-example McNemar's test**, we also need predictions for A and B on the validation sample.

In [ ]:
prompt_A = summary_A["best_prompt_text"]
prompt_B = summary_B["best_prompt_text"]
prompt_C = ga_best_prompt

print("Re-evaluating Experiment A prompt on val split …")
val_A_result = evaluate_prompt(
    prompt=prompt_A,
    examples=val_texts,
    labels=val_labels,
    predict_fn=llm.predict_raw_batch,
    model_id=llm.model_id,
    inference_config=llm.inference_config,
    cache=cache_C,
    split_name="val",
)
pred_A = val_A_result["y_pred"]   # list of int labels
f1_A_val  = val_A_result["metrics"]["f1_macro"]
acc_A_val = val_A_result["metrics"]["accuracy"]
print(f"  Val Macro F1 : {f1_A_val:.4f}")

print("Re-evaluating Experiment B prompt on val split …")
val_B_result = evaluate_prompt(
    prompt=prompt_B,
    examples=val_texts,
    labels=val_labels,
    predict_fn=llm.predict_raw_batch,
    model_id=llm.model_id,
    inference_config=llm.inference_config,
    cache=cache_C,
    split_name="val",
)
pred_B = val_B_result["y_pred"]
f1_B_val  = val_B_result["metrics"]["f1_macro"]
acc_B_val = val_B_result["metrics"]["accuracy"]
print(f"  Val Macro F1 : {f1_B_val:.4f}")

pred_C = ga_val_result["y_pred"]
f1_C_val  = ga_val_f1
acc_C_val = ga_val_acc

val_true = val_labels

## 11. McNemar's Test

McNemar's test measures whether two classifiers make **significantly different mistakes** on the same examples.  
We run three pairwise comparisons: A vs B, A vs C, B vs C.

In [ ]:
def mcnemar_test(y_true, y_pred1, y_pred2, name1="M1", name2="M2", alpha=0.05):
    """Compute McNemar's test between two classifiers.

    Returns a dict with the contingency table, chi2 statistic,
    p-value, and a plain-language verdict.
    """
    # Align lengths — treat INVALID (-1) as a wrong prediction
    n = min(len(y_true), len(y_pred1), len(y_pred2))
    correct1 = np.array([int(y_pred1[i] == y_true[i]) for i in range(n)])
    correct2 = np.array([int(y_pred2[i] == y_true[i]) for i in range(n)])

    # Contingency table:
    #            M2 correct  M2 wrong
    # M1 correct   b          c
    # M1 wrong     a          d  (not used by McNemar)
    b = int(np.sum((correct1 == 1) & (correct2 == 0)))  # M1 right, M2 wrong
    c = int(np.sum((correct1 == 0) & (correct2 == 1)))  # M1 wrong, M2 right

    contingency = np.array([[0, b], [c, 0]])  # McNemar focuses on b and c
    # Use continuity-corrected McNemar (chi2 with df=1)
    bc_sum = b + c
    if bc_sum == 0:
        p_value = 1.0
        chi2    = 0.0
    else:
        chi2  = (abs(b - c) - 1) ** 2 / bc_sum
        from scipy.stats import chi2 as chi2_dist
        p_value = chi2_dist.sf(chi2, df=1)

    significant = p_value < alpha
    if b > c:
        verdict = f"{name1} is significantly better than {name2}" if significant else f"No significant difference between {name1} and {name2}"
    elif c > b:
        verdict = f"{name2} is significantly better than {name1}" if significant else f"No significant difference between {name1} and {name2}"
    else:
        verdict = f"No significant difference between {name1} and {name2}"

    return {
        "comparison": f"{name1} vs {name2}",
        "n_examples":   n,
        "b_m1_right_m2_wrong": b,
        "c_m1_wrong_m2_right": c,
        "chi2":         round(chi2,    6),
        "p_value":      round(p_value, 6),
        "alpha":        alpha,
        "significant":  significant,
        "verdict":      verdict,
    }


mcnemar_AB = mcnemar_test(val_true, pred_A, pred_B, "Exp-A", "Exp-B")
mcnemar_AC = mcnemar_test(val_true, pred_A, pred_C, "Exp-A", "Exp-C (GA)")
mcnemar_BC = mcnemar_test(val_true, pred_B, pred_C, "Exp-B", "Exp-C (GA)")

for mc in [mcnemar_AB, mcnemar_AC, mcnemar_BC]:
    sig_marker = "✓ SIGNIFICANT" if mc["significant"] else "✗ NOT significant"
    print(f"{mc['comparison']:25s}  chi2={mc['chi2']:.3f}  "
          f"p={mc['p_value']:.4f}  [{sig_marker}]")
    print(f"  → {mc['verdict']}")

## 12. Three-Way Comparison Table

In [ ]:
# ── Retrieve opt F1 for A and B from their summaries ─────────────────────────
f1_A_opt  = summary_A["best_macro_f1_opt"]
acc_A_opt = summary_A["best_accuracy_opt"]
f1_B_opt  = summary_B["best_macro_f1_opt"]
acc_B_opt = summary_B["best_accuracy_opt"]

comparison_rows = [
    {
        "experiment": "A",
        "method": "Manual Prompts",
        "n_candidates": summary_A["n_candidates"],
        "evolutionary_pressure": False,
        "best_prompt_id": summary_A["best_prompt_name"],
        "macro_f1_opt":  round(f1_A_opt,  6),
        "accuracy_opt":  round(acc_A_opt, 6),
        "macro_f1_val":  round(f1_A_val,  6),
        "accuracy_val":  round(acc_A_val, 6),
    },
    {
        "experiment": "B",
        "method": "Random Generation",
        "n_candidates": summary_B["n_candidates"],
        "evolutionary_pressure": False,
        "best_prompt_id": summary_B["best_candidate_id"],
        "macro_f1_opt":  round(f1_B_opt,  6),
        "accuracy_opt":  round(acc_B_opt, 6),
        "macro_f1_val":  round(f1_B_val,  6),
        "accuracy_val":  round(acc_B_val, 6),
    },
    {
        "experiment": "C",
        "method": "Genetic Algorithm (seed 42)",
        "n_candidates": 36,   # 12 individuals × 3 generations
        "evolutionary_pressure": True,
        "best_prompt_id": "ga_seed_42_best",
        "macro_f1_opt":  round(ga_opt_f1,  6),
        "accuracy_opt":  round(ga_opt_acc, 6),
        "macro_f1_val":  round(f1_C_val,   6),
        "accuracy_val":  round(acc_C_val,  6),
    },
]

df_compare = pd.DataFrame(comparison_rows).set_index("experiment")
display(df_compare)

# Rank by val Macro F1
ranked = sorted(comparison_rows, key=lambda r: r["macro_f1_val"], reverse=True)
print("\nVal Macro F1 ranking:")
for rank, row in enumerate(ranked, 1):
    print(f"  #{rank} {row['method']:30s}  Val F1 = {row['macro_f1_val']:.4f}")

## 13. Multi-Seed GA Summary (from Member 3's Notebook 06)

In [ ]:
# Member 3 reported: seeds 42, 7, 123
# Best opt Macro F1: 0.943434, 0.941547, 0.941547
# Mean best opt Macro F1: 0.942176
# These values are taken from the official Member 3 handoff message.
GA_MULTI_SEED_RESULTS = [
    {"seed": 42,  "best_macro_f1_opt": 0.943434},
    {"seed": 7,   "best_macro_f1_opt": 0.941547},
    {"seed": 123, "best_macro_f1_opt": 0.941547},
]

# Try to load from Member 3's actual file first; fall back to the values above.
if ga_multi is not None and "runs" in ga_multi:
    print("Using multi-seed data from Member 3's output file.")
else:
    print("Using Member 3 handoff values (multi-seed file not found).")

df_ga_multi = pd.DataFrame(GA_MULTI_SEED_RESULTS)
df_ga_multi["delta_vs_manual_opt"] = (
    df_ga_multi["best_macro_f1_opt"] - f1_A_opt
).round(6)
display(df_ga_multi)

mean_ga_opt = df_ga_multi["best_macro_f1_opt"].mean()
std_ga_opt  = df_ga_multi["best_macro_f1_opt"].std()
print(f"\nGA mean best opt Macro F1 : {mean_ga_opt:.6f}")
print(f"GA std  best opt Macro F1 : {std_ga_opt:.6f}")

## 14. Visualise Three-Way Comparison

In [ ]:
COLORS = {"A": "#3498db", "B": "#9b59b6", "C": "#2ecc71"}
LABELS = {
    "A": "A — Manual",
    "B": "B — Random",
    "C": "C — GA (s42)",
}

exps   = ["A", "B", "C"]
f1_opt = [row["macro_f1_opt"] for row in comparison_rows]
f1_val = [row["macro_f1_val"] for row in comparison_rows]
acc_opt = [row["accuracy_opt"] for row in comparison_rows]
acc_val = [row["accuracy_val"] for row in comparison_rows]
colors = [COLORS[e] for e in exps]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ── Grouped bar: Macro F1 on opt and val ─────────────────────────────────────
x = np.arange(len(exps))
w = 0.35
bars1 = axes[0].bar(x - w/2, f1_opt, w, label="Opt split",  color=colors, alpha=0.60, edgecolor="white")
bars2 = axes[0].bar(x + w/2, f1_val, w, label="Val split",  color=colors, alpha=1.00, edgecolor="white")
axes[0].set_xticks(x)
axes[0].set_xticklabels([LABELS[e] for e in exps])
axes[0].set_ylabel("Macro F1")
axes[0].set_title("Macro F1 — Opt vs Val split\n(best prompt per experiment)")
axes[0].set_ylim(max(0, min(f1_opt + f1_val) - 0.03), min(1, max(f1_opt + f1_val) + 0.03))
axes[0].legend()
for bar, v in zip(list(bars1) + list(bars2), f1_opt + f1_val):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
                 f"{v:.4f}", ha="center", va="bottom", fontsize=8)

# ── GA multi-seed distribution vs baselines ───────────────────────────────────
ga_f1_values = [r["best_macro_f1_opt"] for r in GA_MULTI_SEED_RESULTS]
axes[1].boxplot(ga_f1_values, positions=[1], widths=0.3,
                patch_artist=True,
                boxprops=dict(facecolor=COLORS["C"], alpha=0.7),
                medianprops=dict(color="black", linewidth=2))
axes[1].scatter([1]*len(ga_f1_values), ga_f1_values, color=COLORS["C"],
                zorder=5, s=60)
axes[1].axhline(f1_A_opt, color=COLORS["A"], linestyle="--", linewidth=1.5,
                label=f"Exp-A best opt ({f1_A_opt:.4f})")
axes[1].axhline(f1_B_opt, color=COLORS["B"], linestyle="--", linewidth=1.5,
                label=f"Exp-B best opt ({f1_B_opt:.4f})")
axes[1].set_xticks([1])
axes[1].set_xticklabels(["GA (3 seeds)"])
axes[1].set_ylabel("Best Opt Macro F1")
axes[1].set_title("GA Multi-Seed Distribution\nvs Experiment A and B Baselines")
axes[1].legend(fontsize=8)
axes[1].grid(True, axis="y", alpha=0.3)

plt.tight_layout()
plot_path = OUTPUT_DIR / "three_way_comparison.png"
fig.savefig(plot_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"Figure saved → {plot_path}")

## 15. All Opt-Split Results (A + B together)

This merged leaderboard lets Member 5 see the full distribution of scored candidates.

In [ ]:
rows_A = [
    {"experiment": "A", "method": "manual", "candidate_id": r.get("prompt_name", r.get("candidate_id")),
     "macro_f1_opt": r["macro_f1"], "accuracy_opt": r["accuracy"]}
    for r in summary_A.get("all_opt_results", [])
]
rows_B = [
    {"experiment": "B", "method": "random", "candidate_id": r["candidate_id"],
     "macro_f1_opt": r["macro_f1"], "accuracy_opt": r["accuracy"]}
    for r in summary_B.get("all_opt_results", [])
]
rows_C = [{
    "experiment": "C", "method": "ga_seed42", "candidate_id": "ga_seed_42_best",
    "macro_f1_opt": round(ga_opt_f1, 6), "accuracy_opt": round(ga_opt_acc, 6)
}]

df_all_opt = pd.DataFrame(rows_A + rows_B + rows_C).sort_values(
    "macro_f1_opt", ascending=False
).reset_index(drop=True)
df_all_opt.index += 1
df_all_opt.index.name = "rank"

display(df_all_opt.head(20))
print(f"Total candidates scored: {len(df_all_opt)}")

## 16. Save All Results

In [ ]:
# ── Three-way comparison table ────────────────────────────────────────────────
cmp_csv  = OUTPUT_DIR / "three_way_comparison.csv"
cmp_json = OUTPUT_DIR / "three_way_comparison.json"

df_compare.reset_index().to_csv(cmp_csv, index=False)
with open(cmp_json, "w", encoding="utf-8") as f:
    json.dump(comparison_rows, f, indent=4, ensure_ascii=False, default=str)
print(f"Comparison table → {cmp_csv}")
print(f"                 → {cmp_json}")

# ── McNemar results ───────────────────────────────────────────────────────────
mcnemar_results = [mcnemar_AB, mcnemar_AC, mcnemar_BC]
mcn_path = OUTPUT_DIR / "mcnemar_tests.json"
with open(mcn_path, "w", encoding="utf-8") as f:
    json.dump(mcnemar_results, f, indent=4, ensure_ascii=False, default=str)
print(f"McNemar tests    → {mcn_path}")

# ── GA multi-seed summary ─────────────────────────────────────────────────────
ga_ms_path = OUTPUT_DIR / "ga_multi_seed_summary.json"
with open(ga_ms_path, "w", encoding="utf-8") as f:
    json.dump({
        "runs": GA_MULTI_SEED_RESULTS,
        "mean_best_opt_f1": round(mean_ga_opt, 6),
        "std_best_opt_f1":  round(std_ga_opt,  6),
    }, f, indent=4)
print(f"GA multi-seed    → {ga_ms_path}")

# ── All scored candidates ─────────────────────────────────────────────────────
all_csv = OUTPUT_DIR / "all_opt_candidates.csv"
df_all_opt.reset_index().to_csv(all_csv, index=False)
print(f"All candidates   → {all_csv}")

# ── Master experiment config ──────────────────────────────────────────────────
master_config = {
    "experiment": "C",
    "description": "Full three-way controlled comparison",
    "dataset": "nyu-mll/glue:sst2",
    "model_id": llm.model_id,
    "model_revision": model_config.revision,
    "inference": {"greedy": True, "max_new_tokens": 8, "batch_size": 32},
    "split_seed": 42,
    "evaluation_seed": SEED,
    "n_opt_examples": N_OPT_EXAMPLES,
    "n_val_examples": N_VAL_EXAMPLES,
    "metric_primary": "macro_f1",
    "metric_secondary": "accuracy",
    "test_set_used": False,
    "significance_level": 0.05,
    "significance_test": "McNemar (continuity-corrected, df=1)",
    "experiments_included": ["A", "B", "C"],
    "created_at": datetime.now().isoformat(),
}
master_path = OUTPUT_DIR / "master_experiment_config.json"
master_path.write_text(json.dumps(master_config, indent=4, ensure_ascii=False))
print(f"Master config    → {master_path}")

print("\nAll files saved.")

## 17. Final Summary for Member 5

> **Important**: the numbers below are factual outputs; Member 5 is responsible for the statistical interpretation and research discussion.

In [ ]:
print("=" * 70)
print("THREE-WAY CONTROLLED COMPARISON — FINAL SUMMARY")
print("=" * 70)
print()
print(f"{'Experiment':<35} {'Opt F1':>8} {'Val F1':>8} {'Val Acc':>8}")
print("-" * 60)
for row in comparison_rows:
    print(f"  {row['method']:<33} {row['macro_f1_opt']:>8.4f} "
          f"{row['macro_f1_val']:>8.4f} {row['accuracy_val']:>8.4f}")
print()
print("McNemar pairwise tests (val split, α=0.05):")
for mc in [mcnemar_AB, mcnemar_AC, mcnemar_BC]:
    sig = "SIGNIFICANT" if mc["significant"] else "NOT significant"
    print(f"  {mc['comparison']:25s}  p = {mc['p_value']:.4f}  [{sig}]")
print()
print(f"GA multi-seed (3 runs):")
print(f"  Best opt Macro F1: {[r['best_macro_f1_opt'] for r in GA_MULTI_SEED_RESULTS]}")
print(f"  Mean = {mean_ga_opt:.6f}  |  Std = {std_ga_opt:.6f}")
print()
print("Output directory:", OUTPUT_DIR)
print("Output files:")
for p in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {p.name}")
print()
print("Handoff to Member 5: use three_way_comparison.json and mcnemar_tests.json")
print("for statistical interpretation and research discussion.")